# 🔑 Solutions: Lesson 10 · Project 3 · Crop Disease Classification

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 10.1: Per-channel normalisation  &nbsp; `⭐`

Write `channel_stats(x)` returning `(mean, std)`: each an array of length 3: for images `x` of shape `(N, H, W, 3)`, and `normalise(x, mean, std)` returning the **channels-first** float32 tensor `(N, 3, H, W)` of standardised images.


In [2]:
def channel_stats(x):
    return x.mean(axis=(0, 1, 2)), x.std(axis=(0, 1, 2))

def normalise(x, mean, std):
    return torch.tensor(((x - mean) / std).transpose(0, 3, 1, 2), dtype=torch.float32)

In [3]:
# ▶ self-check
rng = np.random.default_rng(0); x = rng.uniform(0, 1, (50, 8, 8, 3)) * np.array([1, 2, 3])
r = channel_stats(x)
dl.check("mean has 3 entries, matching the data", r is not None and np.allclose(r[0], x.mean((0, 1, 2))))
z = normalise(x, *r) if r is not None else None
dl.check("output shape is (50, 3, 8, 8), float32", z is not None and tuple(z.shape) == (50, 3, 8, 8) and z.dtype == torch.float32)
dl.check("each channel now has mean≈0, std≈1", z is not None and torch.allclose(z.mean((0, 2, 3)), torch.zeros(3), atol=1e-5) and torch.allclose(z.std((0, 2, 3), unbiased=False), torch.ones(3), atol=1e-4))

✅ mean has 3 entries, matching the data
✅ output shape is (50, 3, 8, 8), float32
✅ each channel now has mean≈0, std≈1


---
### 🧩 Problem 10.2: Dihedral augmentation  &nbsp; `⭐⭐`

Write `rot_flip(x, k, flip)` for a batch `x` of shape `(N, C, H, W)`: rotate by `k × 90°` in the spatial plane, then (if `flip`) flip horizontally. Use `torch.rot90` and `Tensor.flip`. Then write `all_8(x)` returning a list of the 8 distinct orientations (k = 0..3 × flip ∈ {False, True}).


In [4]:
def rot_flip(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return x.flip(3) if flip else x

def all_8(x):
    return [rot_flip(x, k, f) for k in range(4) for f in (False, True)]

In [5]:
# ▶ self-check
x = torch.arange(2 * 3 * 4 * 4, dtype=torch.float32).reshape(2, 3, 4, 4)
dl.check("rotating 4 times returns the original", rot_flip(x, 4, False) is not None and torch.equal(rot_flip(x, 4, False), x))
dl.check("rot90 of a (4,4) map matches numpy", rot_flip(x, 1, False) is not None and np.array_equal(rot_flip(x, 1, False)[0, 0].numpy(), np.rot90(x[0, 0].numpy())))
views = all_8(x)
dl.check("8 views", views is not None and len(views) == 8)
dl.check("all 8 views are different", views is not None and len({v[0, 0].numpy().tobytes() for v in views}) == 8)

✅ rotating 4 times returns the original
✅ rot90 of a (4,4) map matches numpy
✅ 8 views
✅ all 8 views are different


---
### 🧩 Problem 10.3: Test-time augmentation  &nbsp; `⭐⭐⭐`

Write `tta_predict(model, x)`: average the **softmax probabilities** of `model` over all 8 orientations of the batch `x` `(N, 3, H, W)` (rotations by 0–270° × optional flip) and return the averaged `(N, num_classes)` probabilities.
Run the model in `eval` mode with gradients off. *(Helper `rot_flip` is provided.)*


In [6]:
def rot_flip(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return x.flip(3) if flip else x

def tta_predict(model, x):
    model.eval()
    total = 0
    with torch.no_grad():
        for k in range(4):
            for flip in (False, True):
                total = total + torch.softmax(model(rot_flip(x, k, flip)), 1)
    return total / 8

In [7]:
# ▶ self-check
# A model built only from global-average pooling is exactly invariant to rotations & flips → TTA must equal plain prediction.
inv = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(3, 4)); inv.eval()
x = torch.randn(5, 3, 8, 8)
out = tta_predict(inv, x)
dl.check("output shape (5, 4), rows sum to 1", out is not None and tuple(out.shape) == (5, 4) and torch.allclose(out.sum(1), torch.ones(5), atol=1e-5))
dl.check("invariant model → TTA equals plain softmax", out is not None and torch.allclose(out, torch.softmax(inv(x), 1), atol=1e-5))
# A model that is NOT invariant should change:
non_inv = nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, 4)); non_inv.eval()
o2 = tta_predict(non_inv, x)
dl.check("non-invariant model → TTA differs from plain prediction", o2 is not None and not torch.allclose(o2, torch.softmax(non_inv(x), 1), atol=1e-3))

✅ output shape (5, 4), rows sum to 1
✅ invariant model → TTA equals plain softmax
✅ non-invariant model → TTA differs from plain prediction


**Why this works.** Averaging over orientations is a free ensemble: errors that depend on a particular orientation tend to cancel.

---
### 🧩 Problem 10.4: Accuracy by severity bin  &nbsp; `⭐⭐`

Write `accuracy_by_bin(correct, values, edges)` where `correct` is a boolean array, `values` the per-sample quantity (e.g. lesion fraction) and `edges` the interior bin boundaries (so `k` edges make `k+1` bins). Return a list with the mean of `correct` in each bin (`nan` for empty bins).


In [8]:
def accuracy_by_bin(correct, values, edges):
    bins = np.digitize(values, edges)
    return [correct[bins == b].mean() if (bins == b).any() else float("nan") for b in range(len(edges) + 1)]

In [9]:
# ▶ self-check
correct = np.array([0, 0, 1, 1, 1, 1], bool); vals = np.array([0.01, 0.02, 0.06, 0.07, 0.3, 0.4])
r = accuracy_by_bin(correct, vals, [0.05, 0.2, 0.5])
dl.check("three populated bins + one empty", r is not None and len(r) == 4)
dl.check("bin 0 accuracy 0.0, bin 1 accuracy 1.0, bin 2 accuracy 1.0", r is not None and r[:3] == [0.0, 1.0, 1.0])
dl.check("empty bin gives NaN", r is not None and np.isnan(r[3]))

✅ three populated bins + one empty
✅ bin 0 accuracy 0.0, bin 1 accuracy 1.0, bin 2 accuracy 1.0
✅ empty bin gives NaN


---
### 🧩 Problem 10.5: Triage the predictions  &nbsp; `⭐⭐`

Write `triage(probs, hi=0.9, lo=0.6)` for a `(N, C)` probability array. Return a dict with the **number** of cases in each tier (`"diagnose"` (max prob ≥ hi), `"retake"` (lo ≤ max prob < hi), `"refer"` (< lo)) which must sum to N.


In [10]:
def triage(probs, hi=0.9, lo=0.6):
    conf = probs.max(axis=1)
    return {"diagnose": int((conf >= hi).sum()), "retake": int(((conf >= lo) & (conf < hi)).sum()), "refer": int((conf < lo).sum())}

In [11]:
# ▶ self-check
p = np.array([[.95, .05], [.7, .3], [.55, .45], [.9, .1], [.6, .4]])
dl.check("{'diagnose': 2, 'retake': 2, 'refer': 1}", triage(p) == {"diagnose": 2, "retake": 2, "refer": 1})
dl.check("stricter thresholds shift cases down", triage(p, hi=0.99) == {"diagnose": 0, "retake": 4, "refer": 1})

✅ {'diagnose': 2, 'retake': 2, 'refer': 1}
✅ stricter thresholds shift cases down


---
### 🧩 Problem 10.6: Occlusion map (challenge)  &nbsp; `⭐⭐⭐`

Write `occlusion_map(model, img, target, patch=8, stride=4)` returning a 2-D NumPy array: entry `(i, j)` is the **drop in the softmax probability of `target`** when the patch of size `patch × patch` at row `i·stride`, column `j·stride` is replaced with zeros.
Output shape = `((H−patch)//stride + 1, (W−patch)//stride + 1)`.


In [12]:
def occlusion_map(model, img, target, patch=8, stride=4):
    model.eval(); _, H, W = img.shape
    with torch.no_grad():
        base = torch.softmax(model(img[None]), 1)[0, target].item()
        ys, xs = range(0, H - patch + 1, stride), range(0, W - patch + 1, stride)
        heat = np.zeros((len(ys), len(xs)))
        for i, y in enumerate(ys):
            for j, x in enumerate(xs):
                occ = img.clone(); occ[:, y:y + patch, x:x + patch] = 0
                heat[i, j] = base - torch.softmax(model(occ[None]), 1)[0, target].item()
    return heat

In [13]:
# ▶ self-check
class CenterModel(nn.Module):
    def forward(self, x):                       # confidence in class 1 depends ONLY on the centre 16×16 region
        c = x[:, :, 8:24, 8:24].mean(dim=(1, 2, 3)) * 5
        return torch.stack([-c, c], dim=1)
m = CenterModel(); img = torch.ones(3, 32, 32)
h = occlusion_map(m, img, 1, patch=8, stride=4)
dl.check("shape is 7×7", h is not None and h.shape == (7, 7))
dl.check("corner patches don't matter (≈0 drop)", h is not None and abs(h[0, 0]) < 1e-6 and abs(h[-1, -1]) < 1e-6)
dl.check("the biggest drop is in the central region", h is not None and 2 <= np.unravel_index(h.argmax(), h.shape)[0] <= 4 and 2 <= np.unravel_index(h.argmax(), h.shape)[1] <= 4)

✅ shape is 7×7
✅ corner patches don't matter (≈0 drop)
✅ the biggest drop is in the central region


**Why this works.** Occlusion is a model-agnostic probe: it only needs the ability to run predictions. The same function works for any image classifier.